**Saving your IBM Quantum Credential.**

Before any notebook can talk to IBM Quantum, two things must be on your
machine: an **API token** and an **instance**. They do different jobs.

- The **API token** is your password. It proves *who* you are.
  It identifies your account.
- An **instance** is an IBM Cloud service instance: the workspace that
  actually runs your circuits. This course uses the `QIS101 - Open` instance,
  which runs on IBM's free Open plan.

Cell 01 writes the token to `~/.qiskit/qiskit-ibm.json` so you never have to
paste it into a notebook again. Cell 02 then connects using that token and
lists the quantum computers available to you.

---
**Cell 01 - Save the credential to disk.**

`save_account()` writes your token to `~/.qiskit/qiskit-ibm.json`.
Run this once. On later runs `AccountAlreadyExistsError` is raised and caught,
which is why `overwrite=False` is safe to leave in place.

**About the `instance` argument.**
The CRN you paste here is stored as your *default* instance, used only when a
notebook asks for no instance by name. Every hardware notebook in this session
names its instance explicitly instead:

```python
service = QiskitRuntimeService(channel="ibm_cloud", instance="QIS101 - Open")
```

So this saved value is a **fallback**. That is deliberate: a default sitting in
a file on your machine is invisible from the code, and a notebook that names
its instance says out loud where it runs and behaves the same on any machine,
whatever happens to be saved there.

To find your CRN: IBM Cloud dashboard -> your Quantum service instance.

**On the redacted output.**
This notebook is committed to git. Printing the raw credential file would
save your API token into the notebook's output and commit it to the
repository, so the token is replaced with a placeholder before printing.
The URL is not a secret, so it prints in full.

In [ ]:
"""qiskit_save_credential.ipynb"""

# Cell 01 - Create the file $HOME/.qiskit/qiskit-ibm.json

import json
from pathlib import Path

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.accounts import AccountAlreadyExistsError

# Attempt to save your IBM Quantum credential
try:
    QiskitRuntimeService.save_account(
        channel="ibm_cloud",  # "channel" distinguishes between different accounts
        instance="PASTE_YOUR_OWN_INSTANCE_CRN_HERE",  # Fallback only - see Cell 02
        set_as_default=True,  # Optionally set these as your default credentials
        token="PASTE_YOUR_OWN_API_TOKEN_HERE",  # Do not share your token in public code
        overwrite=False,  # Do NOT overwrite any existing credential
    )
    print("Created new IBM Qiskit Credential")

except AccountAlreadyExistsError:
    print("Existing IBM Qiskit Credential found")

# Verify the credential file was created.
# This notebook is committed to git, so the token and the account-specific
# part of the CRN are masked before printing - a saved output containing them
# would expose your account to anyone reading the repository.
file_path = Path.home() / ".qiskit" / "qiskit-ibm.json"
with open(file_path, "r") as f_in:
    credentials = json.load(f_in)

print(f"\nFile path: {file_path}")
for account_name, account in credentials.items():
    print(f'\n"{account_name}"')
    for key, value in account.items():
        if key == "token":
            value = f"<masked - {len(str(value))} characters>"
        elif key == "instance":
            value = f"<masked CRN ending ...{str(value)[-14:]}>"
        print(f"    {key}: {value}")

---
**Cell 02 - List the quantum computers available to you.**

This cell connects with the instance named explicitly, then lists the real
devices you can submit to.

`simulator=False` excludes cloud simulators and `operational=True` excludes
machines that are offline or in maintenance, so what you get back is the
hardware you could actually run on right now. These are the same machines
`least_busy()` chooses from in the hardware notebooks.

If the instance name is wrong, this cell fails immediately with
`IBMInputValueError` rather than quietly connecting somewhere else - a loud
error is easier to debug than a silent surprise.

In [ ]:
# Cell 02 - Display available backends

# An IBM Cloud "instance" is the service instance that runs your circuits.
# Name it explicitly so it is always clear which instance a notebook uses,
# rather than relying on the default saved above.
INSTANCE_NAME = "QIS101 - Open"

service = QiskitRuntimeService(channel="ibm_cloud", instance=INSTANCE_NAME)

print(f"Quantum computers available on {INSTANCE_NAME}:\n")
for backend in service.backends(simulator=False, operational=True):
    config = backend.configuration()
    print(
        f"{config.backend_name:15}: Qubits = {config.n_qubits}: Gates = {config.basis_gates}"
    )